# Ursprung Channel Profiler v0.1 — closed-loop walkthrough

Measure `I(secret; observation)` in bits (with CIs), validated against an **analytic anchor**, then close the
loop: over budget → shrink fidelity → re-measure → converge. OBSERVER-class; `estimate ≠ capacity`.

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
for p in (ROOT, os.path.join(ROOT, 'experiments', 'toy_scene')):
    if p not in sys.path: sys.path.insert(0, p)
from analytic_mi import analytic_mi, observation_alphabet_size
print('r  |O|   analytic I(S;O)=H(O) bits  (larger r = more detail = more leakage)')
for r in range(6):
    print(f'{r}  {observation_alphabet_size(radius=r):3d}   {analytic_mi(radius=r):.4f}')

## Validate the estimator against the anchor (r=2, analytic ≈ 1.9722 bits)

In [ ]:
from channel_profiler.estimator import MillerMadowEstimator
from scene import ToyGridScene
sc = ToyGridScene(seed=7, radius=2); est = MillerMadowEstimator(seed=0)
for _ in range(5000): est.ingest(sc.sample_iid())
e = est.estimate()
print(f'verdict={e.verdict}  MI={e.mi_estimate:.4f}  CI=[{e.ci_lower:.4f},{e.ci_upper:.4f}]  n={e.n_samples}')
print(f'analytic = {analytic_mi(radius=2):.4f}  -> covered by CI = {e.ci_lower <= analytic_mi(radius=2) <= e.ci_upper}')

## The closed loop — measure → adapt → re-measure → converge

In [ ]:
import demo_closed_loop as demo
import matplotlib.pyplot as plt
hist = demo.run(); demo._print_summary(hist)
xs=[h['window'] for h in hist]; mi=[h['mi'] for h in hist]
lo=[max(0,m-h['ci_lo']) for m,h in zip(mi,hist)]; hi=[max(0,h['ci_hi']-m) for m,h in zip(mi,hist)]
plt.errorbar(xs, mi, yerr=[lo,hi], fmt='o-', capsize=4, label='measured I(S;O)')
plt.axhline(demo.BUDGET, ls='--', color='r', label=f'budget {demo.BUDGET} bits')
plt.xlabel('estimation window'); plt.ylabel('bits/step'); plt.legend(); plt.title('Closed-loop QIF profiling')
plt.show()